# experiment1

## Reproducibility bootstrap
Run first. Resolves paths for the authors' Drive, a fresh Colab (clones the anon repo), or a local clone. No edits needed.

In [ ]:
# === Reproducibility bootstrap (public bundle) ===
# Resolves all paths for: (a) authors' Google Drive, (b) fresh Colab (clones repo),
# (c) local clone. Sets CODE_DIR, DATA_DIR, RESULTS_DIR, DATA_PATH. Run first; no edits needed.
import os, sys
from pathlib import Path

RESULTS_SUBFOLDER = "experiment1"
DATA_FILENAME = None   # None for synthetic experiments

def _resolve():
    try:
        import google.colab  # noqa: F401
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        dr = Path('/content/drive/MyDrive')
        if (dr/'GNAVAR'/'code'/'gnavar_core.py').exists():
            b = dr/'GNAVAR'; return b/'code', b/'data', b/'results'
        # Fresh Colab without the authors' Drive: the repo files must be present in the
        # session. Anonymous-review repos cannot be git-cloned, so upload the bundle:
        #   1) Download the ZIP from the Anonymous GitHub page (Download / ZIP button).
        #   2) In Colab, upload the ZIP via the Files pane, then in a cell run:
        #        !unzip -o your_bundle.zip
        #   3) %cd into the unzipped repo folder, then run this notebook.
        for cand in [Path('/content')/'ICDM-GNAVAR-EDAE', Path.cwd()]:
            if (cand/'src'/'gnavar_core.py').exists():
                return cand/'src', cand/'data', cand/'results'
        raise FileNotFoundError(
            'Repo files not found in the Colab session. Download the ZIP from the '
            'Anonymous GitHub page, upload and unzip it here, then %cd into the folder '
            'and re-run. See the repository README, Path B, Option B1.')
    except ImportError:
        repo = Path.cwd()
        while repo != repo.parent and not (repo/'verify_paper_numbers.py').exists():
            repo = repo.parent
        return repo/'src', repo/'data', repo/'results'

CODE_DIR, DATA_DIR, RESULTS_ROOT = _resolve()
sys.path.insert(0, str(CODE_DIR))
RESULTS_DIR = RESULTS_ROOT / RESULTS_SUBFOLDER
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = (DATA_DIR / DATA_FILENAME) if DATA_FILENAME else None
DRIVE_ROOT = str(CODE_DIR.parent.parent)  # back-compat for any cell referencing DRIVE_ROOT
print('CODE_DIR    =', CODE_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
if DATA_PATH: print('DATA_PATH   =', DATA_PATH)
import numpy as np
import pandas as pd
import json, hashlib, datetime, time, itertools, platform
try:
    import torch
    import torch.nn as nn
except Exception:
    pass
from gnavar_core import *  # model, generator, fit/eval utils
# === end bootstrap ===


In [ ]:
from gnavar_core import *
import numpy as np, pandas as pd, json, hashlib, datetime


# Experiment 1: Recovery in the Identifiable Regime

Validates Theorem 8.1 of the G-NAVAR identifiability paper by demonstrating that, when the theorem's assumptions hold by construction, modulator sets, gate functions, and edge products are recoverable from a finite trajectory.

## Setup

- **Function class**: G-NAVAR with vector-input gates $g_{ijk}: \mathbb{R}^K \to \mathbb{R}$ matching the paper exactly.
- **DGP**: 5-variable autoregressive system with $K=2$, multi-lag gates (change-based saturation; 2D Gaussian inhibition).
- **Compute**: GPU-vectorized via batched `Conv1d` with groups; mixed precision when CUDA is available.
- **Resilience**: All results write incrementally to Google Drive; runtime disconnects do not lose completed trials.

## Outputs

All outputs go to `/content/drive/MyDrive/GNAVAR/results/experiment1/`:
- `results.csv`: per-trial recovery metrics (incremental)
- `convergence.png`: headline figure (3 panels: modulator-set accuracy, gate $L^2$, edge-product $L^2$ vs $T$)
- `gate_heatmaps.png`: 2D gate recovery visualization at largest $T$
- `summary.txt`: textual summary

## Cell 1: Drive mounting and environment

In [ ]:
# Install any missing dependencies (Colab usually has these already)
try:
    import torch, numpy, pandas, matplotlib
    print(f'torch {torch.__version__} | numpy {numpy.__version__} | pandas {pandas.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'CUDA device: {torch.cuda.get_device_name(0)}')
except ImportError as e:
    print(f'Missing dependency: {e}; installing...')
    !pip install -q torch numpy pandas matplotlib

## Cell 2: Imports and configuration

In [ ]:
import math
import time
import json
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = (DEVICE.type == 'cuda')
print(f'Device: {DEVICE} | Mixed precision: {USE_AMP}')

In [ ]:
@dataclass
class Config:
    # DGP
    n_vars: int = 5            # x1 (target) + x2..x5 (sources)
    K: int = 2                 # lag order
    sigma_eps: float = 0.1     # innovation std for target
    burn_in: int = 200
    ar_coefs: tuple = (0.6, 0.5, 0.7, 0.65)

    # Sweep
    sample_sizes: tuple = (1000, 5000, 25000, 100000)
    n_seeds: int = 5
    base_seed: int = 42

    # Model
    hidden_dim: int = 32

    # Training
    n_epochs: int = 300
    batch_size: int = 512
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5

    # L1 sparsity penalty on gate deviation from 1.
    # Encourages irrelevant gates to collapse to identically-1 (minimality).
    # Tune on a single (T, seed) trial before the full sweep; see lambda-tuning cell below.
    l1_lambda: float = 0.02

    # Minimality threshold: declare g_{ijk} trivial if E[(g - 1)^2] < threshold.
    # Calibrated for L1-trained gates; should be roughly midway between the
    # noise floor of true-trivial gates and the magnitude of true modulators.
    triviality_threshold: float = 0.001

cfg = Config()
print(json.dumps(asdict(cfg), indent=2))

## Cell 3: Data-generating process

**Vector-input multi-lag gates:**
- $g_{123}(x_{3,t-1}, x_{3,t-2}) = \exp(-\tfrac{1}{2}(x_{3,t-1} - x_{3,t-2})^2) \cdot \frac{2}{1 + e^{-2 x_{3,t-1}}}$ — change-sensitive saturation.
- $g_{145}(x_{5,t-1}, x_{5,t-2}) = \exp(-\tfrac{1}{2}(x_{5,t-1}^2 + 0.5 \cdot x_{5,t-2}^2))$ — 2D Gaussian inhibition.

Both gates depend genuinely on both lags of the modulator: fixing one and varying the other produces a non-trivial response.

In [ ]:
def true_f12(x2_lag1):
    return 0.7 * x2_lag1

def true_g123(x3_lag1, x3_lag2):
    """Change-sensitive saturating gate."""
    change_part = np.exp(-0.5 * (x3_lag1 - x3_lag2) ** 2)
    sat_part = 2.0 / (1.0 + np.exp(-2.0 * x3_lag1))
    return change_part * sat_part

def true_f14(x4_lag1):
    return 0.5 * x4_lag1

def true_g145(x5_lag1, x5_lag2):
    """2D Gaussian inhibitory gate, weighted toward the recent lag."""
    return np.exp(-0.5 * (x5_lag1 ** 2 + 0.5 * x5_lag2 ** 2))

def true_f12_lag2(x2_lag2):
    return 0.3 * x2_lag2

In [ ]:
def simulate_dgp(T: int, cfg: Config, seed: int) -> np.ndarray:
    """
    Simulate trajectory of length T (after burn-in).
    Returns shape (T, n_vars). Column 0 is target; columns 1..n_vars-1 are sources.
    """
    rng = np.random.default_rng(seed)
    T_total = T + cfg.burn_in
    X = np.zeros((T_total, cfg.n_vars))
    X[:cfg.K] = 0.1 * rng.standard_normal((cfg.K, cfg.n_vars))

    eps_sources = rng.standard_normal((T_total, cfg.n_vars - 1))
    eps_target = cfg.sigma_eps * rng.standard_normal(T_total)

    for t in range(cfg.K, T_total):
        # Sources: independent AR(1)
        for j in range(cfg.n_vars - 1):
            phi = cfg.ar_coefs[j]
            X[t, j + 1] = phi * X[t - 1, j + 1] + math.sqrt(1 - phi ** 2) * eps_sources[t, j]

        # Target via DGP
        x2_l1, x2_l2 = X[t - 1, 1], X[t - 2, 1]
        x3_l1, x3_l2 = X[t - 1, 2], X[t - 2, 2]
        x4_l1       = X[t - 1, 3]
        x5_l1, x5_l2 = X[t - 1, 4], X[t - 2, 4]

        contrib_23   = true_f12(x2_l1) * true_g123(x3_l1, x3_l2)
        contrib_45   = true_f14(x4_l1) * true_g145(x5_l1, x5_l2)
        contrib_lag2 = true_f12_lag2(x2_l2)

        X[t, 0] = contrib_23 + contrib_45 + contrib_lag2 + eps_target[t]

    return X[cfg.burn_in:]

In [ ]:
def make_lag_tensor(X: np.ndarray, K: int):
    """
    Build supervised pairs (X_lag, y) from a trajectory.

    X_lag: (T - K, n_sources, K). X_lag[t, j, ell] is value of source j+1
           at lag (ell+1) relative to target time (t + K).
    y    : (T - K,) target values at time t + K.
    """
    T, n_vars = X.shape
    n_sources = n_vars - 1
    n_samples = T - K
    X_lag = np.zeros((n_samples, n_sources, K), dtype=np.float32)
    y     = np.zeros(n_samples, dtype=np.float32)
    for t in range(K, T):
        idx = t - K
        y[idx] = X[t, 0]
        for j in range(n_sources):
            for ell in range(K):
                X_lag[idx, j, ell] = X[t - (ell + 1), j + 1]
    return X_lag, y

## Cell 4: Vectorized G-NAVAR model

GPU-friendly implementation. We use `Conv1d` with `groups` to compute many tiny independent MLPs in parallel without Python-level loops.

Architecture for a single target $i$:
- **Bases**: one MLP per source — $f_{ij}: \mathbb{R}^K \to \mathbb{R}$, taking the full lag block of source $j$.
- **Gates**: one MLP per (source, modulator) pair — $g_{ijk}: \mathbb{R}^K \to \mathbb{R}$, taking the full lag block of modulator $k$. Implemented as $g = \exp(\eta(x))$ to be strictly positive.
- For each source $j$, the contribution is $f_{ij}(x_j) \cdot \prod_{k \neq j} g_{ijk}(x_k)$.
- Final prediction is $\beta + \sum_j$ (source contribution).

This is the *vector-input gate* instantiation of the G-NAVAR function class (Section 4.1 of the paper).

In [ ]:
class BatchedMLP(nn.Module):
    """
    A batch of `n_groups` independent small MLPs, each mapping R^K -> R
    via a single hidden layer with Tanh activation.

    Implemented with grouped 1x1 Conv1d so all groups run in parallel on GPU
    without Python loops. Equivalent to having `n_groups` separate Linear layers.

    Input shape : (batch, n_groups, K)
    Output shape: (batch, n_groups)
    """
    def __init__(self, n_groups: int, K: int, hidden_dim: int):
        super().__init__()
        self.n_groups = n_groups
        self.K = K
        self.hidden_dim = hidden_dim
        # First layer: maps n_groups groups, each with K input channels,
        # to n_groups groups, each with hidden_dim output channels.
        self.conv1 = nn.Conv1d(
            in_channels  = n_groups * K,
            out_channels = n_groups * hidden_dim,
            kernel_size  = 1,
            groups       = n_groups,
        )
        # Second layer: maps each group's hidden_dim back to 1.
        self.conv2 = nn.Conv1d(
            in_channels  = n_groups * hidden_dim,
            out_channels = n_groups,
            kernel_size  = 1,
            groups       = n_groups,
        )

    def forward(self, x):
        # x: (batch, n_groups, K)
        batch = x.shape[0]
        # Reshape to (batch, n_groups * K, 1) for grouped conv
        x = x.reshape(batch, self.n_groups * self.K, 1)
        h = torch.tanh(self.conv1(x))           # (batch, n_groups * hidden_dim, 1)
        out = self.conv2(h)                     # (batch, n_groups, 1)
        return out.squeeze(-1)                  # (batch, n_groups)

In [ ]:
class GNAVAR(nn.Module):
    """
    Vectorized G-NAVAR for a single target.

    For n_sources sources and lag order K:
      - n_sources base functions f_j: R^K -> R
      - n_sources * (n_sources - 1) gate functions g_{j,k}: R^K -> R, indexed by
        (source j, modulator k != j).
    Bases and gates are each computed via a single BatchedMLP forward pass.
    """
    def __init__(self, n_sources: int, K: int, hidden_dim: int):
        super().__init__()
        self.n_sources = n_sources
        self.K = K
        self.bias  = nn.Parameter(torch.zeros(1))
        self.bases = BatchedMLP(n_groups=n_sources, K=K, hidden_dim=hidden_dim)
        # Gates: n_sources * (n_sources - 1) per-(j, k) functions.
        self.n_gates = n_sources * (n_sources - 1)
        self.gates   = BatchedMLP(n_groups=self.n_gates, K=K, hidden_dim=hidden_dim)

        # Precompute (j, k) index lookups
        # gate_to_jk[g] = (j, k) for the g-th gate
        # k_index_of_gate[g] = k (used to look up x_k from X_lag)
        # j_index_of_gate[g] = j
        # gate_indices_for_source[j] = list of gate indices whose source is j
        gate_to_jk = []
        for j in range(n_sources):
            for k in range(n_sources):
                if k != j:
                    gate_to_jk.append((j, k))
        self.register_buffer(
            'gate_k_idx',
            torch.tensor([k for (_, k) in gate_to_jk], dtype=torch.long),
        )
        self.register_buffer(
            'gate_j_idx',
            torch.tensor([j for (j, _) in gate_to_jk], dtype=torch.long),
        )

    def forward(self, X_lag):
        """
        X_lag: (batch, n_sources, K)
        Returns: (batch,) prediction.
        """
        batch = X_lag.shape[0]

        # Bases: f_j(x_j) for each source j. Pass X_lag directly; group j sees x_j.
        bases_out = self.bases(X_lag)                      # (batch, n_sources)

        # Gates: gather lag blocks corresponding to each gate's modulator k.
        # gate_k_idx: (n_gates,) — for each gate g, which source k to read.
        gate_inputs = X_lag[:, self.gate_k_idx, :]         # (batch, n_gates, K)
        gate_eta    = self.gates(gate_inputs)              # (batch, n_gates)
        gate_vals   = torch.exp(gate_eta)                  # strictly positive

        # For each source j, multiply its gates together.
        # gate_j_idx tells us which source each gate belongs to.
        # We use scatter-prod via log-sum-exp-style trick to keep it differentiable.
        # log(prod_g gate_vals[g]) = sum_g log(gate_vals[g]) = sum_g eta_g.
        # Sum eta_g over each source j using index_add.
        log_gate_product = torch.zeros(batch, self.n_sources, device=X_lag.device, dtype=gate_eta.dtype)
        log_gate_product = log_gate_product.index_add(
            dim=1,
            index=self.gate_j_idx,
            source=gate_eta,
        )
        gate_product = torch.exp(log_gate_product)         # (batch, n_sources)

        # Per-source contribution and sum
        contributions = bases_out * gate_product           # (batch, n_sources)
        return self.bias + contributions.sum(dim=1)

    @torch.no_grad()
    def evaluate_gate(self, j: int, k: int, x_block):
        """
        Evaluate gate g_{j,k} on input x_block of shape (batch, K).

        Implementation: build a fake input where the gate's modulator slot has
        x_block, others are zero (the gate ignores them by construction).
        Then take the corresponding output channel.
        """
        batch = x_block.shape[0]
        # Find the gate index for (j, k)
        gate_idx = None
        cnt = 0
        for jj in range(self.n_sources):
            for kk in range(self.n_sources):
                if kk != jj:
                    if jj == j and kk == k:
                        gate_idx = cnt
                    cnt += 1
        assert gate_idx is not None, f'Gate ({j},{k}) not found'

        gate_inputs = torch.zeros(batch, self.n_gates, self.K, device=x_block.device, dtype=x_block.dtype)
        gate_inputs[:, gate_idx, :] = x_block
        eta = self.gates(gate_inputs)[:, gate_idx]
        return torch.exp(eta)

    @torch.no_grad()
    def evaluate_base(self, j: int, x_block):
        batch = x_block.shape[0]
        base_inputs = torch.zeros(batch, self.n_sources, self.K, device=x_block.device, dtype=x_block.dtype)
        base_inputs[:, j, :] = x_block
        return self.bases(base_inputs)[:, j]

## Cell 5: Training loop

Adam + mixed precision (when CUDA is available). Data is moved to GPU once at the start of training, never copied per batch.

In [ ]:
def compute_gate_l1_penalty(model, X_lag):
    """
    Compute mean_{batch, gates} |g_{ijk}(x_k) - 1|.

    This is the L1 sparsity penalty: it pushes gates that do not help prediction
    toward identically 1, which corresponds to the minimality condition of the
    theorem. Gates that genuinely matter for prediction will have |g - 1| > 0
    on average (because they actively modulate); irrelevant gates have no reason
    to deviate from 1 once the penalty is in place.
    """
    batch = X_lag.shape[0]
    # Gather lag blocks for each gate's modulator k (same logic as model.forward)
    gate_inputs = X_lag[:, model.gate_k_idx, :]      # (batch, n_gates, K)
    eta = model.gates(gate_inputs)                   # (batch, n_gates)
    g = torch.exp(eta)                               # (batch, n_gates), strictly positive
    return (g - 1.0).abs().mean()                    # scalar


def fit_gnavar(X: np.ndarray, cfg: Config, seed: int, verbose: bool = False) -> GNAVAR:
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)

    n_sources = cfg.n_vars - 1
    X_lag_np, y_np = make_lag_tensor(X, cfg.K)

    X_lag = torch.from_numpy(X_lag_np).to(DEVICE)
    y     = torch.from_numpy(y_np).to(DEVICE)

    model = GNAVAR(n_sources=n_sources, K=cfg.K, hidden_dim=cfg.hidden_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(),
                           lr=cfg.learning_rate,
                           weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    n_samples = X_lag.shape[0]
    for epoch in range(cfg.n_epochs):
        perm = torch.randperm(n_samples, device=DEVICE)
        epoch_mse_sum = 0.0
        epoch_l1_sum  = 0.0
        n_batches = 0
        for i in range(0, n_samples, cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            xb, yb = X_lag[idx], y[idx]
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                pred = model(xb)
                mse  = F.mse_loss(pred, yb)
                l1   = compute_gate_l1_penalty(model, xb)
                loss = mse + cfg.l1_lambda * l1
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_mse_sum += mse.item()
            epoch_l1_sum  += l1.item()
            n_batches += 1
        if verbose and ((epoch + 1) % 50 == 0 or epoch == 0):
            print(f'  epoch {epoch+1:3d}/{cfg.n_epochs}  '
                  f'mse={epoch_mse_sum/n_batches:.5f}  '
                  f'l1={epoch_l1_sum/n_batches:.5f}', flush=True)
    return model

## Cell 6: Recovery metrics

All metrics computed on GPU and brought back to CPU as scalars.

In [ ]:
@torch.no_grad()
def gate_triviality_score(model: GNAVAR, X_lag_t: torch.Tensor, j: int, k: int) -> float:
    """E[(g_{j,k}(x_k) - 1)^2] over the empirical lag-block samples of x_k."""
    n_sources = model.n_sources
    K = model.K
    x_block = X_lag_t[:, k, :]                       # (n_samples, K)
    g = model.evaluate_gate(j=j, k=k, x_block=x_block)
    return float(((g - 1.0) ** 2).mean().item())

@torch.no_grad()
def detect_modulator_set(model: GNAVAR, X_lag_t: torch.Tensor, j: int, threshold: float):
    """Return the set of modulator indices k for which the gate is non-trivial."""
    active = set()
    for k in range(model.n_sources):
        if k == j:
            continue
        if gate_triviality_score(model, X_lag_t, j, k) >= threshold:
            active.add(k)
    return active

@torch.no_grad()
def gauge_normalize(g_vals: np.ndarray):
    """Rescale so empirical mean is 1."""
    m = float(np.mean(g_vals))
    return (g_vals / m, m) if m > 1e-12 else (g_vals, 1.0)

@torch.no_grad()
def gate_l2_error(model: GNAVAR, X_lag_t: torch.Tensor, j: int, k: int,
                  true_gate_fn) -> float:
    """
    L2 error between fitted gate and true gate, evaluated on the empirical
    lag-block samples of x_k. Both are gauge-normalized to E[g] = 1 first.
    true_gate_fn: callable taking (lag1, lag2) numpy arrays.
    """
    x_block = X_lag_t[:, k, :]
    g_fitted = model.evaluate_gate(j=j, k=k, x_block=x_block).cpu().numpy()
    x_block_np = x_block.cpu().numpy()
    g_true = true_gate_fn(x_block_np[:, 0], x_block_np[:, 1])
    g_fitted_n, _ = gauge_normalize(g_fitted)
    g_true_n,   _ = gauge_normalize(g_true)
    return float(np.sqrt(np.mean((g_fitted_n - g_true_n) ** 2)))

@torch.no_grad()
def edge_product_l2_error(model: GNAVAR, X_lag_t: torch.Tensor, X_lag_np: np.ndarray,
                          j: int, true_edge_fn) -> float:
    """
    L2 error between fitted edge product (source j) and true edge product
    over empirical joint support.
    The fitted contribution from source j is f_j(x_j) * prod_k g_{j,k}(x_k).
    """
    n_sources = model.n_sources
    K = model.K
    n_samples = X_lag_t.shape[0]

    # Fitted base for source j
    f_j = model.evaluate_base(j=j, x_block=X_lag_t[:, j, :]).cpu().numpy()
    # Fitted gate product for source j (over k != j)
    log_prod = np.zeros(n_samples)
    for k in range(n_sources):
        if k == j:
            continue
        g_k = model.evaluate_gate(j=j, k=k, x_block=X_lag_t[:, k, :]).cpu().numpy()
        log_prod = log_prod + np.log(np.clip(g_k, 1e-12, None))
    g_prod = np.exp(log_prod)
    fitted = f_j * g_prod

    true_vals = np.array([true_edge_fn(X_lag_np[t]) for t in range(n_samples)])
    return float(np.sqrt(np.mean((fitted - true_vals) ** 2)))

# True edge contribution functions (matching the DGP)
# Layout reminder: X_lag_np[t, j, ell], j in 0..3 (sources x2..x5), ell in 0..1 (lag 1, lag 2)
def true_edge_source0(row):  # source j=0 = x2 ; truly contributes only at lag 1 (modulated)
    x2_l1, x2_l2 = row[0, 0], row[0, 1]
    x3_l1, x3_l2 = row[1, 0], row[1, 1]
    return true_f12(x2_l1) * true_g123(x3_l1, x3_l2)

def true_edge_source2(row):  # source j=2 = x4 ; modulated by x5
    x4_l1 = row[2, 0]
    x5_l1, x5_l2 = row[3, 0], row[3, 1]
    return true_f14(x4_l1) * true_g145(x5_l1, x5_l2)

# Note: the lag-2 unmodulated x2 contribution is captured by source j=0's base function
# operating on the (x2_lag1, x2_lag2) input. In the vector-input fitter, f_j is a single
# function of the full lag block, so the lag-1 modulated and lag-2 unmodulated contributions
# combine into one source-0 contribution. The edge-product metric for source 0 therefore
# evaluates against the *combined* truth: the modulated lag-1 plus the additive lag-2.
def true_edge_source0_combined(row):
    x2_l1, x2_l2 = row[0, 0], row[0, 1]
    x3_l1, x3_l2 = row[1, 0], row[1, 1]
    return true_f12(x2_l1) * true_g123(x3_l1, x3_l2) + true_f12_lag2(x2_l2)

## Cell 7: Resume-aware experiment driver

Each (T, seed) trial is saved as a row to `results.csv` immediately after completion. On rerun, completed trials are skipped — runtime disconnects do not lose finished work.

In [ ]:
RESULTS_CSV = RESULTS_DIR / 'results.csv'

def load_existing_results():
    if RESULTS_CSV.exists():
        df = pd.read_csv(RESULTS_CSV)
        completed = set(zip(df['T'].astype(int), df['seed'].astype(int)))
        return df, completed
    return pd.DataFrame(), set()

def append_result(row: dict):
    """Append a single trial row to results.csv (write header if file is new)."""
    write_header = not RESULTS_CSV.exists()
    pd.DataFrame([row]).to_csv(
        RESULTS_CSV, mode='a', header=write_header, index=False
    )

In [ ]:
def run_trial(T: int, seed: int, cfg: Config, verbose: bool = False) -> dict:
    t0 = time.time()
    X = simulate_dgp(T=T, cfg=cfg, seed=seed)
    model = fit_gnavar(X, cfg, seed=seed + 1000, verbose=verbose)
    model.eval()

    X_lag_np, _ = make_lag_tensor(X, cfg.K)
    X_lag_t = torch.from_numpy(X_lag_np).to(DEVICE)

    # True modulator sets:
    #   source 0 = x2: modulator set = {1} (x3 modulates the lag-1 component)
    #   source 2 = x4: modulator set = {3} (x5 modulates)
    #   source 1 = x3: no contribution -> empty set
    #   source 3 = x5: no contribution -> empty set
    true_M = {0: {1}, 2: {3}}  # only checked for these sources

    detected_M = {
        0: detect_modulator_set(model, X_lag_t, j=0, threshold=cfg.triviality_threshold),
        2: detect_modulator_set(model, X_lag_t, j=2, threshold=cfg.triviality_threshold),
    }
    modulator_correct = int(
        detected_M[0] == true_M[0] and detected_M[2] == true_M[2]
    )

    # Gate L2 error
    err_g123 = gate_l2_error(model, X_lag_t, j=0, k=1, true_gate_fn=true_g123)
    err_g145 = gate_l2_error(model, X_lag_t, j=2, k=3, true_gate_fn=true_g145)

    # Edge product L2 error
    err_edge_source0 = edge_product_l2_error(
        model, X_lag_t, X_lag_np, j=0, true_edge_fn=true_edge_source0_combined
    )
    err_edge_source2 = edge_product_l2_error(
        model, X_lag_t, X_lag_np, j=2, true_edge_fn=true_edge_source2
    )

    elapsed = time.time() - t0
    return {
        'T': T,
        'seed': seed,
        'modulator_correct': modulator_correct,
        'M_source0': sorted(detected_M[0]),
        'M_source2': sorted(detected_M[2]),
        'err_g123': err_g123,
        'err_g145': err_g145,
        'err_edge_source0': err_edge_source0,
        'err_edge_source2': err_edge_source2,
        'elapsed_seconds': elapsed,
    }

In [ ]:
def run_experiment(cfg: Config, verbose: bool = True):
    """Run all (T, seed) trials, skipping any that already exist in results.csv."""
    _, completed = load_existing_results()
    print(f'Already completed: {len(completed)} trials')

    plan = [(T, cfg.base_seed + 1000 * s)
            for T in cfg.sample_sizes
            for s in range(cfg.n_seeds)]
    todo = [(T, sd) for (T, sd) in plan if (T, sd) not in completed]
    print(f'To run: {len(todo)} trials')

    for i, (T, seed) in enumerate(todo, 1):
        print(f'\n[{i}/{len(todo)}] T={T}, seed={seed}', flush=True)
        result = run_trial(T=T, seed=seed, cfg=cfg, verbose=False)
        # Convert sets to strings for CSV compatibility
        row = {**result}
        row['M_source0'] = str(result['M_source0'])
        row['M_source2'] = str(result['M_source2'])
        append_result(row)
        print(f'  done in {result["elapsed_seconds"]:.1f}s | '
              f'modulator_correct={result["modulator_correct"]} | '
              f'err_g123={result["err_g123"]:.4f}, err_g145={result["err_g145"]:.4f}',
              flush=True)

    return load_existing_results()[0]

## Cell 8: Visualization

In [ ]:
def plot_convergence(df: pd.DataFrame, cfg: Config):
    """Headline 3-panel convergence figure."""
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

    # Panel 1: modulator-set accuracy
    g = df.groupby('T')['modulator_correct'].agg(['mean', 'std'])
    axes[0].errorbar(g.index, g['mean'], yerr=g['std'].fillna(0),
                     marker='o', capsize=4, color='C0')
    axes[0].set_xscale('log')
    axes[0].set_ylim(-0.05, 1.05)
    axes[0].set_xlabel('Sample size T')
    axes[0].set_ylabel('Modulator-set recovery accuracy')
    axes[0].set_title('Modulator-set recovery')
    axes[0].grid(True, alpha=0.3)

    # Panel 2: gate L2 error
    df = df.copy()
    df['err_g_avg'] = (df['err_g123'] + df['err_g145']) / 2
    g = df.groupby('T')['err_g_avg'].agg(['mean', 'std'])
    axes[1].errorbar(g.index, g['mean'], yerr=g['std'].fillna(0),
                     marker='o', capsize=4, color='C1')
    axes[1].set_xscale('log'); axes[1].set_yscale('log')
    axes[1].set_xlabel('Sample size T')
    axes[1].set_ylabel(r'Gate $L^2$ error (gauge-normalized)')
    axes[1].set_title('Gate function recovery')
    axes[1].grid(True, alpha=0.3, which='both')

    # Panel 3: edge L2 error
    df['err_edge_avg'] = (df['err_edge_source0'] + df['err_edge_source2']) / 2
    g = df.groupby('T')['err_edge_avg'].agg(['mean', 'std'])
    axes[2].errorbar(g.index, g['mean'], yerr=g['std'].fillna(0),
                     marker='o', capsize=4, color='C2')
    axes[2].set_xscale('log'); axes[2].set_yscale('log')
    axes[2].set_xlabel('Sample size T')
    axes[2].set_ylabel(r'Edge-product $L^2$ error')
    axes[2].set_title('Edge-product recovery')
    axes[2].grid(True, alpha=0.3, which='both')

    fig.suptitle('Experiment 1: Recovery in the identifiable regime', y=1.02)
    fig.tight_layout()
    out = RESULTS_DIR / 'convergence.png'
    fig.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')

In [ ]:
def plot_gate_heatmaps(cfg: Config):
    """
    Side-by-side heatmaps of fitted vs true gate at the largest T, single seed.

    Both true and fitted gates are gauge-normalized using the same empirical-lag-block
    samples that the metric uses (not a uniform grid), so the visual magnitudes match
    the L2 errors reported in the convergence figure.
    """
    T = cfg.sample_sizes[-1]
    seed = cfg.base_seed
    print(f'Refitting at T={T}, seed={seed} for visualization...')
    X = simulate_dgp(T=T, cfg=cfg, seed=seed)
    model = fit_gnavar(X, cfg, seed=seed + 1000, verbose=False)
    model.eval()

    # Empirical lag-block samples for x3 (modulator of edge 0) and x5 (modulator of edge 2)
    X_lag_np, _ = make_lag_tensor(X, cfg.K)
    x3_samples = torch.from_numpy(X_lag_np[:, 1, :]).to(DEVICE)  # (n_samples, K)
    x5_samples = torch.from_numpy(X_lag_np[:, 3, :]).to(DEVICE)

    # Grid for visualization
    axis_vals = np.linspace(-3, 3, 60).astype(np.float32)
    L1, L2 = np.meshgrid(axis_vals, axis_vals, indexing='ij')
    grid_np = np.stack([L1.ravel(), L2.ravel()], axis=1)  # (3600, 2)
    grid_t = torch.from_numpy(grid_np).to(DEVICE)

    with torch.no_grad():
        # Gate values on empirical samples (used for gauge normalization)
        g123_fit_samples = model.evaluate_gate(j=0, k=1, x_block=x3_samples).cpu().numpy()
        g145_fit_samples = model.evaluate_gate(j=2, k=3, x_block=x5_samples).cpu().numpy()
        # Gate values on visualization grid
        g123_fit_grid = model.evaluate_gate(j=0, k=1, x_block=grid_t).cpu().numpy().reshape(60, 60)
        g145_fit_grid = model.evaluate_gate(j=2, k=3, x_block=grid_t).cpu().numpy().reshape(60, 60)

    # True gates: evaluate on the same empirical samples AND on the grid
    x3_samples_np = X_lag_np[:, 1, :]
    x5_samples_np = X_lag_np[:, 3, :]
    g123_true_samples = true_g123(x3_samples_np[:, 0], x3_samples_np[:, 1])
    g145_true_samples = true_g145(x5_samples_np[:, 0], x5_samples_np[:, 1])
    g123_true_grid = true_g123(L1, L2)
    g145_true_grid = true_g145(L1, L2)

    # Gauge-normalize using EMPIRICAL means (matches the metric convention).
    # The same scalar factor is applied to grid-evaluated gates so the heatmap
    # magnitude reflects the gauge under which the L2 error was computed.
    def _norm(grid, samples):
        m = float(np.mean(samples))
        if m <= 1e-12: return grid
        return grid / m

    g123_fit_grid  = _norm(g123_fit_grid,  g123_fit_samples)
    g145_fit_grid  = _norm(g145_fit_grid,  g145_fit_samples)
    g123_true_grid = _norm(g123_true_grid, g123_true_samples)
    g145_true_grid = _norm(g145_true_grid, g145_true_samples)

    fig, axes = plt.subplots(2, 2, figsize=(10, 9))
    extent = (axis_vals[0], axis_vals[-1], axis_vals[0], axis_vals[-1])

    # Share color scale within each row so true vs fitted are directly comparable
    for row_idx, (true_mat, fit_mat, title_true, title_fit) in enumerate([
        (g123_true_grid, g123_fit_grid,
         r'True $g_{123}$ (saturating $\times$ change)',  r'Fitted $\hat{g}_{123}$'),
        (g145_true_grid, g145_fit_grid,
         r'True $g_{145}$ (2D Gaussian inhibition)',       r'Fitted $\hat{g}_{145}$'),
    ]):
        vmin = min(true_mat.min(), fit_mat.min())
        vmax = max(true_mat.max(), fit_mat.max())
        for col_idx, (mat, title) in enumerate([(true_mat, title_true), (fit_mat, title_fit)]):
            ax = axes[row_idx, col_idx]
            im = ax.imshow(mat.T, origin='lower', extent=extent, aspect='auto',
                           cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_xlabel(r'$x_{k, t-1}$ (lag 1)')
            ax.set_ylabel(r'$x_{k, t-2}$ (lag 2)')
            ax.set_title(title)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle(f'Gate recovery (T={T}, seed={seed}): fitted vs true (shared row colorbar)', y=1.00)
    fig.tight_layout()
    out = RESULTS_DIR / 'gate_heatmaps.png'
    fig.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')

In [ ]:
def write_summary(df: pd.DataFrame, cfg: Config):
    lines = [
        'Experiment 1: Recovery in the identifiable regime',
        '=' * 60,
        f'Sample sizes: {list(cfg.sample_sizes)}',
        f'Replications: {cfg.n_seeds}',
        f'Total trials: {len(df)}',
        '',
        'Aggregated by sample size (mean +/- std across seeds):',
        '',
    ]
    for T in cfg.sample_sizes:
        sub = df[df['T'] == T]
        if len(sub) == 0:
            continue
        lines.append(f'T = {T}:')
        lines.append(f'  modulator_correct: {sub["modulator_correct"].mean():.3f} +/- {sub["modulator_correct"].std():.3f}')
        lines.append(f'  err_g123:          {sub["err_g123"].mean():.4f} +/- {sub["err_g123"].std():.4f}')
        lines.append(f'  err_g145:          {sub["err_g145"].mean():.4f} +/- {sub["err_g145"].std():.4f}')
        lines.append(f'  err_edge_source0:  {sub["err_edge_source0"].mean():.4f} +/- {sub["err_edge_source0"].std():.4f}')
        lines.append(f'  err_edge_source2:  {sub["err_edge_source2"].mean():.4f} +/- {sub["err_edge_source2"].std():.4f}')
        lines.append(f'  mean wall time:    {sub["elapsed_seconds"].mean():.1f}s')
        lines.append('')
    summary = '\n'.join(lines)
    out = RESULTS_DIR / 'summary.txt'
    out.write_text(summary)
    print(summary)
    print(f'\nSaved {out}')

## Cell 8b: L1 sparsity penalty tuning (run this first)

Before the full sweep, run a single (T=25000, seed=42) trial across a small
set of `l1_lambda` values to pick the one that gives clean modulator-set
recovery on synthetic data. Use the chosen value in `cfg.l1_lambda` for the
full sweep below.

What to look for:
- **All gates trivial** (modulator sets empty for both edges): lambda too high
- **Many spurious modulators detected**: lambda too low
- **Correct modulators ({1} for edge 0, {3} for edge 2), small gate L2 error**: just right

In [ ]:
def tune_l1_lambda(cfg: Config, lambda_values=(0.005, 0.01, 0.02, 0.05, 0.1),
                   T_tune: int = 25000, tune_seed: int = 42):
    """
    Sweep l1_lambda at a fixed (T, seed) trial and report diagnostics.
    Does not write to results.csv (separate file).
    """
    print(f'Tuning l1_lambda at T={T_tune}, seed={tune_seed}\n')
    print(f'{"lambda":>8}  {"M_src0":>10}  {"M_src2":>10}  '
          f'{"err_g123":>10}  {"err_g145":>10}  {"correct":>8}')
    print('-' * 70)
    rows = []
    for lam in lambda_values:
        cfg_tune = Config(**{**asdict(cfg), 'l1_lambda': lam})
        result = run_trial(T=T_tune, seed=tune_seed, cfg=cfg_tune, verbose=False)
        print(f'{lam:>8.4f}  {str(result["M_source0"]):>10}  {str(result["M_source2"]):>10}  '
              f'{result["err_g123"]:>10.4f}  {result["err_g145"]:>10.4f}  '
              f'{result["modulator_correct"]:>8}')
        rows.append({'l1_lambda': lam, **result})
    df_tune = pd.DataFrame(rows)
    df_tune.to_csv(RESULTS_DIR / 'l1_tuning.csv', index=False)
    print(f'\nSaved tuning results to {RESULTS_DIR / "l1_tuning.csv"}')
    return df_tune

df_tune = tune_l1_lambda(cfg)

## Cell 9: Run experiment

Run all trials. Re-running this cell after a disconnect picks up from where it stopped.

In [ ]:
df = run_experiment(cfg, verbose=True)
print(f'\nAll trials complete. Total rows: {len(df)}')

In [ ]:
plot_convergence(df, cfg)

In [ ]:
plot_gate_heatmaps(cfg)

In [ ]:
write_summary(df, cfg)